# Text2Outfit: midway runs (Colab A100)

Results needed for the midway report:
1. Replication table: CP AUC and FITB accuracy of OutfitTransformer (CLIP) on the Polyvore nondisjoint test split.
2. Main table, B1 row: zero-shot FashionCLIP retrieval.
3. Compute table: wall-clock time of each step (see the `%%time` cells).

| Step | Task | Estimate |
|---|---|---|
| 0 | Setup, mount Drive | 5 min |
| 1 | Data download (~2.2 GB) | 5-10 min |
| 2 | FashionCLIP embeddings (251k items) | 20-40 min |
| 3 | CP training | 1-2 h |
| 4 | CP test, FITB | 5 min |
| 5 | B1 baseline | 5-10 min |
| 6 | CIR training + FITB (optional) | 30-60 min |
| 7 | Judge model, second seed (optional) | 1-2 h |

Before running: upload the `outfit-transformer` folder to Google Drive and set `DRIVE_ROOT` below.
Runtime type: A100 GPU, High-RAM.

## Step 0: Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
!nproc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/10623'
DRIVE_REPO = f'{DRIVE_ROOT}/outfit-transformer'
DRIVE_OUT  = f'{DRIVE_ROOT}/midway_outputs'
REPO = '/content/outfit-transformer'

import os
os.makedirs(DRIVE_OUT, exist_ok=True)
!rm -rf {REPO} && cp -r "{DRIVE_REPO}" {REPO}
%cd {REPO}
!ls src/run

In [ ]:
# Colab already ships torch, transformers and datasets.
!pip -q install pydantic wandb faiss-cpu
import torch, transformers, datasets
print(torch.__version__, transformers.__version__, datasets.__version__, torch.cuda.get_device_name(0))

## Step 1: Data

Item metadata and images (`owj0421/polyvore`) are downloaded on first import.
Outfit titles come from `mvasil/polyvore-outfits`, which requires accepting the dataset terms on Hugging Face and logging in.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%%time
from huggingface_hub import hf_hub_download
import shutil, json
p = hf_hub_download('mvasil/polyvore-outfits', 'polyvore_outfit_titles.json', repo_type='dataset')
TITLES = f'{DRIVE_OUT}/polyvore_outfit_titles.json'
shutil.copy(p, TITLES)
t = json.load(open(TITLES)); k = next(iter(t)); print(len(t), k, t[k])

In [ ]:
%%time
!cd {REPO} && python -c "
from src.data.datasets import polyvore as P
d = P.PolyvoreCompatibilityDataset(dataset_type='nondisjoint', dataset_split='valid')
print('CP valid', len(d)); x = d[0]; print(x['label'], [(i.item_id, i.category) for i in x['query'].outfit])
f = P.PolyvoreFillInTheBlankDataset(dataset_type='nondisjoint', dataset_split='test')
print('FITB test', len(f))
"

## Step 2: FashionCLIP embeddings

Output: `datasets/polyvore/precomputed_clip_embeddings/polyvore_0.pkl` (~1 GB), copied to Drive.
After a disconnect, run the restore cell instead.

In [ ]:
%%time
!cd {REPO} && python -m src.run.1_generate_clip_embeddings --demo --batch_sz_per_gpu 64

In [ ]:
%%time
!cd {REPO} && python -m src.run.1_generate_clip_embeddings --batch_sz_per_gpu 256 --n_workers_per_gpu 8
!ls -lh {REPO}/datasets/polyvore/precomputed_clip_embeddings/
!mkdir -p "{DRIVE_OUT}/precomputed_clip_embeddings" && cp {REPO}/datasets/polyvore/precomputed_clip_embeddings/*.pkl "{DRIVE_OUT}/precomputed_clip_embeddings/"

In [ ]:
# Restore embeddings from Drive
!mkdir -p {REPO}/datasets/polyvore/precomputed_clip_embeddings && cp "{DRIVE_OUT}"/precomputed_clip_embeddings/*.pkl {REPO}/datasets/polyvore/precomputed_clip_embeddings/

## Step 3: Compatibility prediction (CP) training

The repository default is 200 epochs. The OneCycle schedule scales with `--n_epochs`.
Check the time of the first epoch, then choose `N_EPOCHS` (50 if an epoch takes about 1.5 min or less, otherwise 30).

In [ ]:
%%time
!cd {REPO} && python -m src.run.2_train_compatibility --demo --n_epochs 1 --project_name smoke --n_workers_per_gpu 8

In [ ]:
N_EPOCHS = 50
!mkdir -p /content/runlogs
!cd {REPO} && nohup python -m src.run.2_train_compatibility --n_epochs {N_EPOCHS} --seed 42 --project_name seed42 --n_workers_per_gpu 8 > /content/runlogs/cp_seed42.log 2>&1 &

In [ ]:
# Progress
!grep -E "End|Best|Error|Traceback" {REPO}/logs/compatibility_clip_seed42_rank0.log | tail -n 8
!tail -c 400 /content/runlogs/cp_seed42.log

In [ ]:
import json
CP_DIR = f'{REPO}/checkpoints/compatibility_clip_seed42'
best = json.load(open(f'{CP_DIR}/best.json')); print(best)
CP_CKPT = f"{CP_DIR}/epoch_{best['best_epoch']}.pth"
!mkdir -p "{DRIVE_OUT}/checkpoints" && cp {CP_CKPT} {CP_DIR}/*.json "{DRIVE_OUT}/checkpoints/"
!ls -lh "{DRIVE_OUT}/checkpoints"

## Step 4: CP AUC and FITB

In [ ]:
%%time
!cd {REPO} && python -m src.run.2_test_compatibility --checkpoint {CP_CKPT}

In [ ]:
%%time
!cd {REPO} && python -m src.run.2_test_fitb_cp --checkpoint {CP_CKPT}

## Step 5: Baseline B1 (zero-shot FashionCLIP)

Query = outfit title; top-1/2/3 item per slot gives M = 3 outfits.
B1 does not use the CP model for selection, so the replicated CP model is used as the judge here.

In [ ]:
%%time
!cd {REPO} && python -m src.run.4_baseline_b1 --titles_json "{TITLES}" --judge_checkpoint {CP_CKPT} --n_queries 2000 --M 3 --K 10
!cp -r {REPO}/results "{DRIVE_OUT}/"

## Step 6 (optional): CIR training and FITB

Warm-started from the best CP checkpoint.

In [ ]:
!cd {REPO} && nohup python -m src.run.3_train_complementary --checkpoint {CP_CKPT} --n_epochs 20 --project_name seed42 --n_workers_per_gpu 8 > /content/runlogs/cir_seed42.log 2>&1 &

In [ ]:
!grep -E "End|Best|Error|Traceback" {REPO}/logs/complementary_clip_seed42_rank0.log | tail -n 6

In [ ]:
CIR_DIR = f'{REPO}/checkpoints/complementary_clip_seed42'
bc = json.load(open(f'{CIR_DIR}/best.json')); print(bc)
CIR_CKPT = f"{CIR_DIR}/epoch_{bc['best_epoch']}.pth"
!cd {REPO} && python -m src.run.3_test_complementary --checkpoint {CIR_CKPT}
!cp {CIR_CKPT} "{DRIVE_OUT}/checkpoints/cir_best.pth"; cp -r {REPO}/results "{DRIVE_OUT}/"

## Step 7 (optional): Judge model

Same as Step 3 with a different seed and port. Can run alongside Step 6.

In [ ]:
!cd {REPO} && MASTER_PORT=12356 nohup python -m src.run.2_train_compatibility --n_epochs {N_EPOCHS} --seed 7 --project_name judge_seed7 --n_workers_per_gpu 6 > /content/runlogs/cp_judge.log 2>&1 &

## Summary

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{REPO}/results/**/*.json', recursive=True)):
    if 'examples' in f: continue
    print(f.replace(REPO + '/', ''), '\n   ', json.load(open(f)))